# MTurk Task A Mining

This notebook mines **all** matching pre-spike / post-spike Task A examples and writes them to CSV/JSON.
The later `selecter.ipynb` notebook filters these down to 100 rows with model-balanced quotas.
For Task A, the selector only judges the **post-spike** row, then keeps the matching pre-spike row with it.


In [ ]:

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path('/playpen-ssd/smerrill/deception2')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from mturk_dataset_utils import (
    DATASETS_ROOT,
    ENVIRONMENTS,
    MODEL_VARIANTS,
    MTURK_CACHE_ROOT,
    MTURK_OUTPUT_ROOT,
    build_taska_dataframe,
    save_task_dataframe,
    taska_summary_dataframe,
)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)


In [ ]:

# Configuration
MAX_EXAMPLES_PER_ENV_TO_LOAD = None  # Use None to mine every localization file
LARGE_SPIKE_DELTA_THRESHOLD = 0.50
RECENT_SENTENCES_TO_SHOW = 3
CACHE_ROOT = MTURK_CACHE_ROOT / 'taska_mining'
REFRESH_MINING_CACHE = False
PROGRESS_EVERY_FILES = 250
WRITE_JSON_EXPORT = True

CSV_PATH = MTURK_OUTPUT_ROOT / 'taska.csv'
JSON_PATH = MTURK_OUTPUT_ROOT / 'taska.json'
SUMMARY_PATH = MTURK_OUTPUT_ROOT / 'taska_summary.csv'

print('Configuration')
print(f'  Dataset root: {DATASETS_ROOT}')
print(f'  Output root: {MTURK_OUTPUT_ROOT}')
print(f'  Cache root: {CACHE_ROOT}')
print(f'  Models: {list(MODEL_VARIANTS.keys())}')
print(f'  Environments: {ENVIRONMENTS}')
print(f'  Max examples per env: {MAX_EXAMPLES_PER_ENV_TO_LOAD}')
print(f'  Spike threshold: {LARGE_SPIKE_DELTA_THRESHOLD}')
print(f'  Recent sentences shown: {RECENT_SENTENCES_TO_SHOW}')
print(f'  Refresh mining cache: {REFRESH_MINING_CACHE}')
print(f'  Progress every files: {PROGRESS_EVERY_FILES}')
print(f'  Write JSON export: {WRITE_JSON_EXPORT}')


In [ ]:

taska_df = build_taska_dataframe(
    dataset_root=DATASETS_ROOT,
    model_variants=MODEL_VARIANTS,
    environments=ENVIRONMENTS,
    threshold=LARGE_SPIKE_DELTA_THRESHOLD,
    recent_sentences_to_show=RECENT_SENTENCES_TO_SHOW,
    max_examples_per_env_to_load=MAX_EXAMPLES_PER_ENV_TO_LOAD,
    cache_root=CACHE_ROOT,
    refresh_cache=REFRESH_MINING_CACHE,
    progress_every_files=PROGRESS_EVERY_FILES,
)
taska_summary_df = taska_summary_dataframe(taska_df)

print(f'Mined {len(taska_df)} Task A rows across {taska_df["pair_id"].nunique() if not taska_df.empty else 0} pairs')
display(taska_summary_df)
display(taska_df.head(10))


In [ ]:
if not taska_df.empty:
    preview_cols = [
        'task_id', 'pair_id', 'pair_role', 'model_id', 'environment', 'example_id',
        'sentence_idx', 'spike_sentence_idx', 'reasoning_snippet',
        'pair_gold_action_value', 'pair_gold_action_share', 'spike_delta',
    ]
    display(taska_df[preview_cols].head(20))
else:
    print('No Task A rows found with the current settings.')


In [ ]:

save_task_dataframe(
    taska_df,
    csv_path=CSV_PATH,
    json_path=JSON_PATH if WRITE_JSON_EXPORT else None,
    summary_df=taska_summary_df,
    summary_path=SUMMARY_PATH,
)

print(f'Saved CSV to:     {CSV_PATH}')
if WRITE_JSON_EXPORT:
    print(f'Saved JSON to:    {JSON_PATH}')
else:
    print('Skipped JSON export')
print(f'Saved summary to: {SUMMARY_PATH}')
